# D009 종합 프로젝트 — 빵셀러 end-to-end 정제·변환·저장 파이프라인

원본 판매 로그(CSV) → **품질 점검** → **정제 5단계(.pipe)** → **KPI 변환** → **Parquet 저장**까지 한 번에 수행합니다.

| 산출물 | 경로 |
|---|---|
| 파이프라인 모듈 | `pipeline.py` |
| 정제 데이터 | `outputs/bakery_clean.parquet` |
| 환불 분리 데이터 | `outputs/bakery_refunds.parquet` |
| 월·매장 KPI | `outputs/bakery_store_month.parquet` |
| 상품별 KPI | `outputs/bakery_item_kpi.parquet` |

In [1]:
import os
import time
from pathlib import Path

import pandas as pd

from pipeline import (
    quality_report_full, log_decision,
    drop_duplicates, strip_store_id, normalize_item, split_refunds, drop_missing_revenue,
    store_month_kpi, item_kpi, load_raw, full_pipeline,
)

RAW_PATH = Path("data/bakery_raw.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

if not RAW_PATH.exists():  # 원본이 없으면 재현 스크립트로 생성 (seed 고정)
    from data.make_bakery_raw import make_raw
    make_raw().to_csv(RAW_PATH, index=False, encoding="utf-8-sig")

pd.set_option("display.float_format", "{:,.0f}".format)

## 1. 원본 데이터 적재

In [2]:
raw = load_raw(RAW_PATH)
print(raw.shape)
raw.head()

(30300, 7)


,order_id,order_date,store_id,item,qty,unit_price,revenue
0,O021130,2025-05-03,S01,식빵,2,4500,"9,000"
1,O015934,2025-05-20,S02,스콘,5,3300,"16,500"
2,O007506,2025-06-02,S01,케이크,4,28000,"112,000"
3,O028526,2025-06-07,S02,케이크,4,28000,"112,000"
4,O019439,2025-05-18,S03,크루아상,1,3800,"3,800"


## 2. 품질 점검 — `quality_report_full()`

컬럼별 타입·결측·고유값·음수·공백 개수를 한 표로 확인합니다.

In [3]:
pd.set_option("display.float_format", "{:,.2f}".format)
report = quality_report_full(raw)
print(f"전체 행: {report.attrs['n_rows']:,} / 완전 중복 행: {report.attrs['n_duplicated']:,}")
report

전체 행: 30,300 / 완전 중복 행: 300


,dtype,n_missing,missing_pct,n_unique,n_negative,n_whitespace,min,max,sample
column,,,,,,,,,
order_id,object,0,0.00,30000,NaN,0.00,NaN,NaN,"O021130, O015934, O007506, O028526, O019439"
order_date,datetime64[ns],0,0.00,91,NaN,NaN,NaN,NaN,"2025-05-03 00:00:00, 2025-05-20 00:00:00, 2025..."
store_id,object,0,0.00,15,NaN,"1,216.00",NaN,NaN,"S01, S02, S03, S04, S05"
item,object,0,0.00,9,NaN,0.00,NaN,NaN,"식빵, 스콘, 케이크, 크루아상, 베이글"
qty,int64,0,0.00,5,0.00,NaN,1.00,5.00,"2, 5, 4, 1, 3"
unit_price,int64,0,0.00,8,0.00,NaN,"2,500.00","28,000.00","4500, 3300, 28000, 3800, 3500"
revenue,float64,450,1.49,78,240.00,NaN,"-140,000.00","140,000.00","9000.0, 16500.0, 112000.0, 3800.0, 7000.0"


In [4]:
# 표기 혼재 확인: item 값 분포
raw["item"].value_counts()

item
소금빵     6014
식빵      5446
크루아상    4553
베이글     3731
마카롱     3690
스콘      3007
바게트     2340
케이크      989
케익       530
Name: count, dtype: int64

In [5]:
# store_id 표기 확인: 공백 때문에 매장이 5개가 아니라 더 많이 보인다
raw["store_id"].map(repr).value_counts()

store_id
'S01'     8675
'S02'     7299
'S03'     5929
'S04'     4250
'S05'     2931
'S01 '     175
' S01'     170
' S02'     152
'S02 '     149
' S03'     142
'S03 '     118
'S04 '      89
' S04'      80
'S05 '      75
' S05'      66
Name: count, dtype: int64

**점검 결과 요약**
- 결측: `revenue` 1.49% (450건)
- 중복: 완전 중복 300건
- 이상치: `revenue` 음수 240건 (최솟값 -140,000원) → 환불 거래
- 표기 혼재: `store_id` 앞뒤 공백 1216건, `item` '케익' 530건 ('케이크'와 동일 상품)

※ 원본 기준 건수입니다. 결정 로그의 건수는 중복 제거 이후 기준이라 조금 작습니다.

## 3. 정제 5단계 — 작은 함수 + `.pipe()`

각 단계 함수는 `pipeline.py`에 있고, **함수 내부에서 `log_decision()`을 자동 호출**해 결정 로그를 쌓습니다.

| 단계 | 함수 | 처리 |
|---|---|---|
| 1 | `drop_duplicates` | 완전 중복 행 제거 |
| 2 | `strip_store_id` | store_id 앞뒤 공백 제거 |
| 3 | `normalize_item` | '케익' → '케이크' |
| 4 | `split_refunds` | revenue < 0 행을 별도 테이블로 분리 |
| 5 | `drop_missing_revenue` | revenue 결측 행 제거 |

In [6]:
import inspect
print(inspect.getsource(split_refunds))

def split_refunds(df: pd.DataFrame, log: list, side: dict) -> pd.DataFrame:
    is_refund = df["revenue"] < 0
    side["refunds"] = df[is_refund].copy()
    out = df[~is_refund]
    log_decision(log, "4. 환불 분리", "revenue < 0 → 별도 테이블(refunds)",
                 len(df), len(out), int(is_refund.sum()),
                 "오류가 아니라 환불 거래. 삭제하지 않고 따로 보관해 후속 분석에 사용")
    return out



In [7]:
decision_log, side = [], {}

bakery_clean = (
    raw
    .pipe(drop_duplicates, log=decision_log)
    .pipe(strip_store_id, log=decision_log)
    .pipe(normalize_item, log=decision_log)
    .pipe(split_refunds, log=decision_log, side=side)
    .pipe(drop_missing_revenue, log=decision_log)
    .assign(month=lambda d: d["order_date"].dt.to_period("M").astype(str))
    .reset_index(drop=True)
)
refunds = side["refunds"]

print(f"원본 {len(raw):,}행 → 정제 후 {len(bakery_clean):,}행 (환불 {len(refunds):,}행 별도 보관)")

원본 30,300행 → 정제 후 29,312행 (환불 238행 별도 보관)


In [8]:
# 정제 후 재점검: 결측·중복·음수·공백이 모두 0이어야 한다
after = quality_report_full(bakery_clean)
print("완전 중복:", after.attrs["n_duplicated"])
after[["dtype", "n_missing", "n_negative", "n_whitespace", "n_unique"]]

완전 중복: 0


,dtype,n_missing,n_negative,n_whitespace,n_unique
column,,,,,
order_id,object,0,NaN,0.00,29312
order_date,datetime64[ns],0,NaN,NaN,91
store_id,object,0,NaN,0.00,5
item,object,0,NaN,0.00,8
qty,int64,0,0.00,NaN,5
unit_price,int64,0,0.00,NaN,8
revenue,float64,0,0.00,NaN,39
month,object,0,NaN,0.00,3


In [9]:
assert after.attrs["n_duplicated"] == 0
assert bakery_clean["revenue"].isna().sum() == 0
assert (bakery_clean["revenue"] < 0).sum() == 0
assert sorted(bakery_clean["store_id"].unique()) == ["S01", "S02", "S03", "S04", "S05"]
assert "케익" not in set(bakery_clean["item"])
print("정제 검증 통과")

정제 검증 통과


## 4. 결정 로그

In [10]:
decision_log_df = pd.DataFrame(decision_log)
print(decision_log_df.to_markdown(index=False))

| step                  | rule                               |   rows_before |   rows_after |   affected | reason                                                                                                      |
|:----------------------|:-----------------------------------|--------------:|-------------:|-----------:|:------------------------------------------------------------------------------------------------------------|
| 1. 중복 제거          | 완전 중복 행 drop_duplicates()     |         30300 |        30000 |        300 | 같은 order_id·내용이 그대로 반복 = 로그 재전송. 매출 이중 집계 방지                                         |
| 2. store_id 공백 제거 | str.strip()                        |         30000 |        30000 |       1200 | ' S01'과 'S01'이 다른 매장으로 groupby 되는 문제 방지                                                       |
| 3. 상품명 표기 통일   | '케익' → '케이크'                  |         30000 |        30000 |        524 | 같은 상품이 두 줄로 쪼개져 상품 KPI가 과소 집계되는 문제 방지                                               |
|

결정 로그 표 (위 `to_markdown()` 출력을 그대로 옮김)

| step                  | rule                               |   rows_before |   rows_after |   affected | reason                                                                                                      |
|:----------------------|:-----------------------------------|--------------:|-------------:|-----------:|:------------------------------------------------------------------------------------------------------------|
| 1. 중복 제거          | 완전 중복 행 drop_duplicates()     |         30300 |        30000 |        300 | 같은 order_id·내용이 그대로 반복 = 로그 재전송. 매출 이중 집계 방지                                         |
| 2. store_id 공백 제거 | str.strip()                        |         30000 |        30000 |       1200 | ' S01'과 'S01'이 다른 매장으로 groupby 되는 문제 방지                                                       |
| 3. 상품명 표기 통일   | '케익' → '케이크'                  |         30000 |        30000 |        524 | 같은 상품이 두 줄로 쪼개져 상품 KPI가 과소 집계되는 문제 방지                                               |
| 4. 환불 분리          | revenue < 0 → 별도 테이블(refunds) |         30000 |        29762 |        238 | 오류가 아니라 환불 거래. 삭제하지 않고 따로 보관해 후속 분석에 사용                                         |
| 5. 매출 결측 제거     | revenue NaN 행 dropna()            |         29762 |        29312 |        450 | qty×unit_price로 복원은 가능하나 결제 실패 여부를 알 수 없어 매출로 확정하지 않음 (1.5% 수준이라 영향 작음) |

## 5. 변환 — KPI 테이블

In [11]:
pd.set_option("display.float_format", "{:,.0f}".format)
store_month = store_month_kpi(bakery_clean)
store_month

,month,store_id,revenue,orders,qty,avg_order_value
0,2025-04,S01,"42,218,600",2920,8791,"14,458"
1,2025-04,S02,"34,023,700",2415,7227,"14,088"
2,2025-04,S03,"28,039,100",1942,5862,"14,438"
3,2025-04,S04,"19,489,000",1404,4253,"13,881"
4,2025-04,S05,"14,140,100",989,2956,"14,297"
5,2025-05,S01,"43,636,500",2971,9026,"14,687"
6,2025-05,S02,"36,604,300",2496,7435,"14,665"
7,2025-05,S03,"27,106,800",2019,5984,"13,426"
8,2025-05,S04,"21,768,200",1427,4289,"15,255"
9,2025-05,S05,"15,132,300",1014,3080,"14,923"


In [12]:
# 월 × 매장 매출 피벗 (단위: 원)
store_month.pivot(index="store_id", columns="month", values="revenue")

month,2025-04,2025-05,2025-06
store_id,,,
S01,"42,218,600","43,636,500","42,489,000"
S02,"34,023,700","36,604,300","33,773,400"
S03,"28,039,100","27,106,800","29,576,200"
S04,"19,489,000","21,768,200","20,913,200"
S05,"14,140,100","15,132,300","14,350,400"


In [13]:
pd.set_option("display.float_format", "{:,.2f}".format)
item_kpi_df = item_kpi(bakery_clean)
item_kpi_df

,item,revenue,qty,orders,revenue_share_pct,rank
0,케이크,"125,664,000.00",4488,1461,29.69,1
1,식빵,"70,681,500.00",15707,5283,16.70,2
2,소금빵,"56,326,400.00",17602,5818,13.31,3
3,크루아상,"50,049,800.00",13171,4403,11.82,4
4,베이글,"38,279,500.00",10937,3622,9.04,5
5,스콘,"28,485,600.00",8632,2895,6.73,6
6,바게트,"26,944,000.00",6736,2252,6.37,7
7,마카롱,"26,830,000.00",10732,3578,6.34,8


## 6. 저장 — Parquet vs CSV

In [14]:
bakery_clean.to_parquet(OUT_DIR / "bakery_clean.parquet", index=False)
bakery_clean.to_csv(OUT_DIR / "bakery_clean.csv", index=False, encoding="utf-8-sig")
refunds.to_parquet(OUT_DIR / "bakery_refunds.parquet", index=False)
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet", index=False)
item_kpi_df.to_parquet(OUT_DIR / "bakery_item_kpi.parquet", index=False)

pq_kb = os.path.getsize(OUT_DIR / "bakery_clean.parquet") / 1024
csv_kb = os.path.getsize(OUT_DIR / "bakery_clean.csv") / 1024
print(f"CSV {csv_kb:,.0f}KB → Parquet {pq_kb:,.0f}KB ({csv_kb / pq_kb:.1f}배 작음)")

CSV 1,609KB → Parquet 303KB (5.3배 작음)


In [15]:
# Parquet은 dtype까지 보존한다 (CSV는 다시 읽으면 order_date가 문자열)
print(pd.read_parquet(OUT_DIR / "bakery_clean.parquet").dtypes["order_date"])
print(pd.read_csv(OUT_DIR / "bakery_clean.csv").dtypes["order_date"])
(OUT_DIR / "bakery_clean.csv").unlink()  # 비교용 CSV는 저장소에 올리지 않음

datetime64[ns]

object


> **용량 비교:** 같은 bakery_clean이 CSV 1,609KB → Parquet 303KB로 약 **5.3배** 작고, 날짜 같은 dtype도 그대로 보존됩니다.

## 7. (심화) 모듈 호출 — `from pipeline import full_pipeline`

위 1~6단계 전체를 한 줄로 실행하고, 단계별 실행 결과와 동일한지 확인합니다.

In [16]:
result = full_pipeline(RAW_PATH, OUT_DIR)

pd.testing.assert_frame_equal(result["clean"], bakery_clean)
pd.testing.assert_frame_equal(result["store_month"], store_month)
pd.testing.assert_frame_equal(result["item_kpi"], item_kpi_df)
print("full_pipeline() 결과 = 단계별 결과 ✔")
sorted(p.name for p in OUT_DIR.glob("*.parquet"))

full_pipeline() 결과 = 단계별 결과 ✔


['bakery_clean.parquet',
 'bakery_item_kpi.parquet',
 'bakery_refunds.parquet',
 'bakery_store_month.parquet']

## 8. (심화) Polars로 같은 파이프라인 작성 + 속도 비교

CSV 읽기부터 KPI 2종 계산까지를 각각 5회 실행한 중앙값으로 비교합니다.

In [17]:
import polars as pl


def pandas_run():
    df = load_raw(RAW_PATH)
    log, side = [], {}
    c = (df.pipe(drop_duplicates, log=log).pipe(strip_store_id, log=log)
           .pipe(normalize_item, log=log).pipe(split_refunds, log=log, side=side)
           .pipe(drop_missing_revenue, log=log)
           .assign(month=lambda d: d["order_date"].dt.to_period("M").astype(str)))
    return store_month_kpi(c), item_kpi(c)


def polars_run():
    c = (
        pl.read_csv(RAW_PATH, try_parse_dates=True, schema_overrides={"store_id": pl.Utf8})
        .unique(maintain_order=True)
        .with_columns(pl.col("store_id").str.strip_chars(),
                      pl.col("item").replace({"케익": "케이크"}))
        .filter(~(pl.col("revenue") < 0))
        .drop_nulls("revenue")
        .with_columns(pl.col("order_date").dt.strftime("%Y-%m").alias("month"))
    )
    sm = (c.group_by(["month", "store_id"])
           .agg(pl.col("revenue").sum(), pl.col("order_id").n_unique().alias("orders"),
                pl.col("qty").sum())
           .with_columns((pl.col("revenue") / pl.col("orders")).round(0).alias("avg_order_value"))
           .sort(["month", "store_id"]))
    it = (c.group_by("item")
           .agg(pl.col("revenue").sum(), pl.col("qty").sum(),
                pl.col("order_id").n_unique().alias("orders"))
           .sort("revenue", descending=True))
    return sm, it


def bench(fn, n=5):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return sorted(times)[n // 2] * 1000


# 두 도구 결과가 같은지 먼저 확인
pd_sm, pd_it = pandas_run()
pl_sm, pl_it = polars_run()
assert len(pl_sm) == len(pd_sm)
assert pl_sm["revenue"].to_list() == pd_sm["revenue"].tolist()
assert pl_it["item"].to_list() == pd_it["item"].tolist()

pd_ms, pl_ms = bench(pandas_run), bench(polars_run)
speed = pd.DataFrame({"tool": ["pandas", "polars"], "median_ms": [pd_ms, pl_ms]})
speed["vs_pandas"] = (pd_ms / speed["median_ms"]).round(1)
speed

,tool,median_ms,vs_pandas
0,pandas,83.05,1.00
1,polars,12.53,6.60


> 약 3만 행 규모에서는 두 도구 모두 0.1초 안팎이라 체감 차이는 작지만, 이번 실행에서 Polars가 약 5배 빨랐습니다. Polars는 CSV 파싱과 group_by를 멀티스레드로 처리해 더 빠르고, 데이터가 커질수록 격차가 커집니다. 다만 단계별 결정 로그·`.pipe()` 재사용성은 기존 pandas 코드가 더 편해서 이번 과제의 기준 구현은 pandas로 유지했습니다.

# 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그 (`data/make_bakery_raw.py`, seed=42로 재현 가능)
- 기간: 2025-04-01 ~ 2025-06-30 (3개월), 매장 5곳(S01~S05), 상품 8종
- 원본 행 수 / 정제 후 행 수: **30,300행 → 29,312행** (환불 238행은 `bakery_refunds.parquet`에 별도 보관)

## 2. 발견한 품질 문제
- 결측: `revenue` 컬럼 1.49% (450건), 나머지 컬럼은 결측 없음
- 중복: 완전 중복 행 300건 (로그 재전송으로 같은 주문이 두 번 기록됨)
- 이상치: `revenue` 음수 240건 (최솟값 -140,000원) — 금액이 정확히 qty×unit_price의 음수라 입력 오류가 아니라 환불
- 표기 혼재: `store_id` 앞뒤 공백 1216건(4.01%, 예: `' S01'`, `'S02 '`), `item` '케익' 530건이 '케이크'와 별도 상품으로 집계됨

## 3. 처리 결정과 근거 (5줄로)
_(건수는 앞 단계 처리 후 기준 — 결정 로그와 동일)_
1. 중복 행 300건 → 제거 (같은 주문의 이중 기록이라 남기면 매출이 부풀려짐)
2. store_id 공백 1200건 → `str.strip()`으로 제거 (공백 때문에 매장 5곳이 15개 그룹으로 쪼개짐)
3. item '케익' → '케이크' 524건 통일 (같은 상품이 두 줄로 나뉘어 상품 KPI가 과소 집계됨)
4. revenue 음수 → 별도 보관 238건 (오류가 아니라 환불 거래라 삭제하지 않고 `bakery_refunds.parquet`로 분리)
5. revenue 결측 450건 → 제거 (이유: qty×unit_price로 값은 복원할 수 있지만 결제가 실제로 완료됐는지 알 수 없어 매출로 확정하지 않음, 전체의 1.49%라 KPI 영향 작음)

## 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: **S01 / 2025-05**, 매출 43,636,500원 (주문 2,971건)
- (상품 관점) 매출 1위 상품: **케이크** 125,664,000원 (전체 매출의 29.69%) — 판매 수량은 적지만 단가가 높아 1위

## 5. 한계와 후속 작업
- 환불 238건의 상품·매장·월별 발생 원인 분석 필요 (원 주문과 매칭할 키가 없어 순매출 계산 불가)
- revenue 결측 450건이 결제 실패인지 단순 누락인지 결제 시스템 로그와 대조 필요 (누락이면 복원)
- item '케익'/'케이크' 외에 다른 표기 변종(띄어쓰기, 영문 표기 등)이 있는지 신규 데이터 유입 시 자동 점검 필요
- 데이터가 가상 생성 데이터라 실제 계절성·요일 효과는 반영되지 않음